# **Esercitazione su object detection**
Nell'esercitazione odierna utilizzeremo l'architettura YOLO per rilevare i pezzi degli scacchi presenti nelle immagini del [Chess Pieces Dataset](https://public.roboflow.com/object-detection/chess-full).

# **Operazioni preliminari**
Eseguendo la cella sottostante tutto il materiale necessario per lo svolgimento dell'esercitazione verrà scaricato sulla macchina remota. Alla fine dell'esecuzione selezionare il tab **Files** per verificare che tutto sia stato scaricato correttamente.

In [ ]:
!wget http://bias.csr.unibo.it/VR/Esercitazioni/DBs/ObjectDetection/ChessPiecesDataset_416x416_aug.zip

!unzip -q /content/ChessPiecesDataset_416x416_aug.zip

!rm /content/ChessPiecesDataset_416x416_aug.zip

## **Ultralytics**
In questa esercitazione faremo uso della libreria [**ultralytics**](https://pypi.org/project/ultralytics/), che offre una suite di modelli e strumenti per affrontare diversi task di *computer vision*, tra cui:  *object detection*, *tracking*, *instance segmentation*, *image classification* e *pose estimation*.

Eseguendo la cella seguente verrà installata l'ultima versione della libreria **ultralytics**.

In [ ]:
!uv pip install ultralytics

# **Import delle librerie**
Come primo passo, è necessario importare le librerie che verranno utilizzate nel corso dell'esercitazione.

In [ ]:
import os
import numpy as np
import random
import math
import pandas as pd
from ultralytics import YOLO
from matplotlib import pyplot as plt

# **Functioni di utilità**
La cella seguente definisce alcune funzioni di utilità che verranno utilizzate nel corso dell'esercitazione:
- **extract_history** estrae dai risultati restituiti dal modello le loss e le metriche calcolate durante la fase di addestramento;
- **plot_history** visualizza l'andamento delle loss e della *mean Average Precision* (mAP) sui dataset di training e validation;
- **plot_results** visualizza i risultati ottenuti dal modello sovrapponendo alle immagini le *bounding box* rilevate;
- **plot_images_with_xywh_bounding_boxes** visualizza una lista di immagini, sovrapponendo a ciascuna le relative *bounding box*;
- **parse_yolo_annotation** legge da un file di testo le annotazioni associate a un'immagine nel formato YOLO.

In [ ]:
def extract_history(model):
  df = pd.read_csv(model.trainer.csv)

  box_w=model.trainer.args.box
  cls_w=model.trainer.args.cls
  dfl_w=model.trainer.args.dfl

  history=  {
              "train_loss" : box_w*df["train/box_loss"] + cls_w*df["train/cls_loss"] + dfl_w*df["train/dfl_loss"],
              "val_loss" : box_w*df["val/box_loss"] + cls_w*df["val/cls_loss"] + dfl_w*df["val/dfl_loss"],
              "val_precision" : df["metrics/precision(B)"],
              "val_recall" : df["metrics/recall(B)"],
              "val_mAP50" : df["metrics/mAP50(B)"],
              "val_mAP50-95" : df["metrics/mAP50-95(B)"],
            }

  return history

def plot_history(history):
  fig, ax1 = plt.subplots(figsize=(10, 8))

  epoch_count=len(history["train_loss"])

  line1,=ax1.plot(range(1,epoch_count+1),history["train_loss"],label="train_loss",color="orange")
  ax1.plot(range(1,epoch_count+1),history["val_loss"],label="val_loss",color = line1.get_color(), linestyle = '--')
  ax1.set_xlim([1,epoch_count])
  ax1.set_ylim([0, max(max(history["train_loss"]),max(history["val_loss"]))])
  ax1.set_ylabel("loss",color = line1.get_color())
  ax1.tick_params(axis="y", labelcolor=line1.get_color())
  ax1.set_xlabel("Epochs")
  _=ax1.legend(loc="lower left")

  ax2 = ax1.twinx()
  line2,=ax2.plot(range(1,epoch_count+1),history["val_mAP50"],label="val_mAP50")
  ax2.set_ylim([0, 1])
  ax2.set_ylabel("metrics",color=line2.get_color())
  ax2.tick_params(axis="y", labelcolor=line2.get_color())
  _=ax2.legend(loc="upper right")

def plot_results(results,image_per_row=4,show_labels=True):
  image_count=len(results)
  row_count=math.ceil(image_count/image_per_row)
  col_count=image_per_row

  _, axs = plt.subplots(nrows=row_count, ncols=col_count,figsize=(18, 4*row_count),squeeze=False)
  for r in range(row_count):
      for c in range(col_count):
        axs[r,c].axis("off")

  for i in range(image_count):
    r = i // image_per_row
    c = i % image_per_row

    axs[r,c].imshow(results[i].plot(labels=show_labels))

def plot_images_with_xywh_bounding_boxes(images,boxes,class_ids,class_labels,image_per_row=4,show_labels=True,confidences=None):
  class_colors = plt.cm.hsv(np.linspace(0, 1, len(class_labels)+1)).tolist()

  image_count=len(images)
  row_count=math.ceil(image_count/image_per_row)
  col_count=image_per_row

  _, axs = plt.subplots(nrows=row_count, ncols=col_count,figsize=(18, 4*row_count),squeeze=False)
  for r in range(row_count):
      for c in range(col_count):
        axs[r,c].axis("off")

  for i in range(image_count):
    r = i // image_per_row
    c = i % image_per_row

    axs[r,c].imshow(images[i])
    image_height=images[i].shape[0]
    image_width=images[i].shape[1]
    for box_idx in range(len(boxes[i])):
        box=boxes[i][box_idx]
        class_idx=class_ids[i][box_idx]
        color =class_colors[class_idx]
        xmin=box[0]*image_width
        ymin=box[1]*image_height
        w=box[2]*image_width
        h=box[3]*image_height
        axs[r,c].add_patch(plt.Rectangle((xmin,ymin), w, h, color=color, fill=False, linewidth=2))
        if show_labels:
          label ="{}".format(class_labels[class_idx])
          if confidences is not None:
            label+=" {:.2f}".format(confidences[i][box_idx])
          axs[r,c].text(xmin, ymin, label, size="large", color="white", bbox={"facecolor":color, "alpha":1.0})

def parse_yolo_annotation(txt_file):
    yolo_boxes = []
    class_ids = []
    with open(txt_file) as file:
      for line in file:
        splitted_line = line.split()
        class_ids.append(int(splitted_line[0]))
        rcx = float(splitted_line[1])
        rcy = float(splitted_line[2])
        rw = float(splitted_line[3])
        rh = float(splitted_line[4])
        rxmin=rcx-rw/2
        rymin=rcy-rh/2
        yolo_boxes.append([rxmin, rymin, rw, rh]) #rel_xywh

    return yolo_boxes, class_ids

#YOLO11
[YOLO11](https://docs.ultralytics.com/models/yolo11/) è l'ultima evoluzione dell'architettura YOLO (*You Only Look Once*) progettata per rilevare in tempo reale oggetti multipli all'interno di un'immagine con elevata precisione ed efficienza. È adatto a una vasta gamma di task di *computer vision*, tra cui *object detection*, *instance segmentation*, *image classification*, *pose estimation* e *oriented object detection*.

## **Creazione del modello**
Il codice seguente crea un'istanza della classe [YOLO](https://docs.ultralytics.com/reference/models/yolo/model/), che fornisce un'interfaccia unificata per l'utilizzo dei diversi [modelli](https://docs.ultralytics.com/models/) basati sull'architettura YOLO. Nel nostro caso, verrà istanziao il modello YOLO11. Esistono diverse [versioni](https://docs.ultralytics.com/tasks/detect/#models) di YOLO11, ciascuna con caratteristiche e dimensioni differenti. Considerate le limitazioni in termini di tempo e risorse hardware, in questa esercitazione utilizzeremo la versione *nano* (*n*), ottimizzata per la velocità e l'efficienza su dispositivi con capacità computazionale ridotta..  


In [ ]:
yolo_detector=YOLO("yolo11n.yaml")

È possibile istanziare un modello YOLO con pesi pre-addestrati sul dataset [COCO](https://cocodataset.org/) 2017, che contiene immagini RGB annotate con oggetti appartenenti a 80 classi.

<img src=https://biolab.csr.unibo.it/vr/esercitazioni/NotebookImages/EsObjectDetection\COCO2017Classes.png width="800">

Il codice seguente crea un'istanza del modello YOLO11 *nano* con pesi pre-addestrati. Con questa configurazione, il modello è in grado di raggiungere una *mean Average Precision* (mAP) pari a 0.55 sul dataset di validazione COCO 2017.


In [ ]:
yolo_detector=YOLO("yolo11n.pt")

## **Visualizzazione del modello**
Eseguendo la cella seguente è possibile stampare un riepilogo testuale della struttura del modello appena caricato.

In [ ]:
yolo_detector.info(detailed=True)

# **Esercizio 1**
Utilizzare il modello appena istanziato per rilevare gli oggetti presenti in una o più immagini scaricate dal web.

A tal fine:

1. scaricare una o più immagini dal web contenenti istanze delle classi presenti nel dataset COCO 2017;
2. caricare le immagini su **Colab** utilizzando la funzione *Upload* nel tab **Files**;
3. leggere le immagini e inserirle nella lista *images* (per farlo utilizzare la funzione [**imread**](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imread.html) della libreria [**Matplotlib**](https://matplotlib.org/));
4. effettuare la *detection* degli oggetti richiamando il metodo [**predict**](https://docs.ultralytics.com/reference/engine/model/#ultralytics.engine.model.Model.predict) del modello utilizzando la variabile *images* come input e memorizzando l'output nella variabile *results*.

In [ ]:
# Soluzione:

#Punto 3
images = #...

# Punto 4
results = #...

## **Output del modello**
Il modello restituisce una lista contenente un'istanza della classe [**Results**](https://docs.ultralytics.com/reference/engine/results/#ultralytics.engine.results.Results) per ciascun input fornito al metodo **predict**.

Tra i vari attributi disponibili all'interno della classe **Results**, quello di nostro interesse è l'attributo *boxes*, di tipo [**Boxes**](https://docs.ultralytics.com/reference/engine/results/#ultralytics.engine.results.Boxes). Questo attributo raccoglie, per ogni immagine analizzata e per ciascun oggetto rilevato:
- le coordinate della *bounding box* che racchiude l'oggetto,
- la classe di appartenenza dell'oggetto rilevato,
- il valore di confidenza associato alla predizione.

In [ ]:
print("Lunghezza della lista results: ",len(results))
print("Oggetti rilevati nella prima immagine: ",len(results[0].boxes))
print("Coordinate del primo oggetto rilevato nella prima immagine: ",results[0].boxes[0].xywh)
print("Classe del primo oggetto rilevato nella prima immagine: ",yolo_detector.names[int(results[0].boxes[0].cls)])
print("Confidenza di predizione del primo oggetto rilevato nella prima immagine: ",results[0].boxes[0].conf)

## **Visualizzazione del risultato**
Richiamando la funzione di utilità **plot_results**, è possibile visualizzare gli oggetti rilevati dal modello, sovrapponendo le bounding box alle immagini originali.

In [ ]:
plot_results(results,image_per_row=2)

# **Dataset**
In questa esercitazione utilizzeremo il [Chess Pieces Dataset](https://public.roboflow.com/object-detection/chess-full) per addestrare il modello YOLO11 a riconoscere i pezzi del gioco degli scacchi.

Il dataset è composto da un insieme di immagini RGB, ciascuna accompagnata da un file di annotazioni in formato testuale (.txt). Ogni file contiene, per ciascun pezzo presente nell'immagine, le coordinate della bounding box (*ground truth*) e la classe di appartenenza.

Il formato con cui sono memorizzate le *bounding box* dei pezzi degli scacchi riporta $x$, $y$, $w$ e $h$ come valori relativi (nell'intervallo $[0;1]$) rispetto alla dimensione dell'immagine a cui sono associate.

La cella sottostante stampa a video i valori relativi $(x,y,w,h)$ delle *bounding box* associate alla prima immagine del training set, insieme alle corrispondenti classi di appartenenza.

In [ ]:
label_file_path = "/content/416x416_aug/train/labels/00bc0cacffdebe6b11bdeec56f63ee49_jpg.rf.1a1407058a6170f001f2c269411d31d3.txt"

yolo_boxes, class_ids=parse_yolo_annotation(label_file_path)

print("Valori (x, y, w, h) in formato relativo:", yolo_boxes)
print("Classi degli oggetti presenti:", class_ids)

## **Visualizzazione dei dati**
Per comprendere meglio la natura e la complessità del problema che affronteremo, può essere molto utile osservare alcune immagini di esempio.
Eseguendo la cella sottostante verranno visualizzate le prime quattro immagini del training set, ciascuna accompagnata dalle relative annotazioni (bounding box e classi).

In [ ]:
# Classi del problema
chess_class_labels = ["none",
                      "black-bishop", "black-king", "black-knight", "black-pawn", "black-queen", "black-rook",
                      "white-bishop", "white-king", "white-knight", "white-pawn", "white-queen", "white-rook"]

image_files = [f for f in os.listdir("/content/416x416_aug/train/images") if f.endswith((".jpg", ".jpeg", ".png"))]
image_paths = [os.path.join("/content/416x416_aug/train/images", f) for f in image_files[:4]]
image_basenames = [os.path.splitext(os.path.basename(p))[0] for p in image_paths]
label_paths = [os.path.join("/content/416x416_aug/train/labels", f + ".txt") for f in image_basenames]

boxes=[]
class_ids=[]
for label_path in label_paths:
  curr_boxes, curr_ids=parse_yolo_annotation(label_path)
  boxes.append(curr_boxes)
  class_ids.append(curr_ids)

plot_images_with_xywh_bounding_boxes([plt.imread(p) for p in image_paths],
                                     boxes,
                                     class_ids,
                                     chess_class_labels,
                                     image_per_row=4,
                                     show_labels=False)

# **Transfer learning**
L'addestramento di YOLOv11 su un nuovo problema richiede un training set etichettato di dimensioni significative, in grado di rappresentare adeguatamente la varietà e la complessità degli oggetti da rilevare.

In alternativa all'addestramento da zero (*training from scratch*), è possibile partire da un modello pre-addestrato su un altro task di *object detection* (ad esempio su COCO 2017) ed effettuare un [*fine-tuning*](https://en.wikipedia.org/wiki/Fine-tuning_(deep_learning)) sul nostro specifico problema. In questo modo è possibile ridurre i tempi di addestramento e migliorare la qualità dei risultati, soprattutto quando il dataset disponibile è limitato.


## **Caricamento del modello pre-addestrato**
Il codice seguente crea un'istanza del modello YOLO11 *nano* con pesi pre-addestrati sul dataset COCO 2017.

In [ ]:
yolo_detector=YOLO("yolo11n.pt")

# **Training**
Ora siamo pronti per effettuare il *fine-tuning* del modello utilizzando le immagini e le annotazioni dei dataset di training e validation per un numero di epoche pari a *epoch_count*.

Per evitare che il modello vada incontro a fenomeni di *overfitting* sul dataset di training, è possibile utilizzare il parametro *patience*, che consente di interrompere automaticamente l'addestramento quando la loss calcolata sul validation set smette di migliorare per un certo numero di epoche consecutive.

In [ ]:
# Numero di epoche di addestramento
epoch_count = 30

# Numero di epoche consecutive senza miglioramenti dopo le quali l'addestramento verrà interrotto
patience=10

La cella sottostante richiama il metodo [**train**](https://docs.ultralytics.com/reference/engine/model/#ultralytics.engine.model.Model.train), che esegue l'intera fase di addestramento in modo automatico. Durante il processo, viene monitorata costantemente la *loss function* sui dataset di training e validation, oltre alla *mean Average Precision* (mAP).

Per fornire al modello i dati di training e validation, è sufficiente impostare il parametro *data* con il percorso del file di configurazione del dataset, che specifica la struttura delle cartelle e le classi da rilevare.


In [ ]:
yolo_detector.train(data="416x416_aug/data.yaml",epochs=epoch_count,patience=patience,cache=True,verbose=False)

Grazie all'utilizzo delle funzioni di utilità **extract_history** e **plot_history**, è possibile visualizzare, per ogni epoca, l'andamento della *loss function* sui dataset di training e validation, oltre ai valori della mAP.

In [ ]:
history=extract_history(yolo_detector)

plot_history(history)

È possibile personalizzare numerosi iperparametri del metodo **train**, come ad esempio l'ottimizzatore, il *learning rate* e il *momentum*. Per farlo, è sufficiente consultare l'elenco completo dei parametri di input disponibili a questo [link](https://docs.ultralytics.com/modes/train/#train-settings).

# **Valutazione delle prestazioni sul test set**
Per verificare l'effettiva capacità di generalizzazione del modello appena addestrato, è fondamentale valutarne le prestazioni sul dataset di test.

## **Object detection**
Eseguendo la cella sottostante, verrà effettuata la *detection* sulle immagini del test set utilizzando il metodo **predict** del modello appena addestrato. In questo caso è sufficiente fornire come input il percorso della cartella contenente le immagini del test set.

È importante notare che, prima di poter richiamare il metodo **predict**, è necessario impostare il modello in modalità *evaluation* utilizzando il metodo [**eval**](https://docs.ultralytics.com/reference/engine/model/#ultralytics.engine.model.Model.eval).
Questo passaggio consente di disattivare i meccanismi specifici della fase di addestramento (come il *dropout* o la *batch normalization*), garantendo che il modello operi correttamente durante la fase di inferenza.

In [ ]:
yolo_detector.eval()

results=yolo_detector.predict(source="416x416_aug/test/images")

## **Visualizzazione dei risultati**
Richiamando la funzione di utilità **plot_results**, è possibile visualizzare i pezzi degli scacchi rilevati dal modello, sovrapponendo le bounding box alle immagini originali.

In [ ]:
plot_results(results,image_per_row=5,show_labels=False)

Visivamente si possono notare gli ottimi risultati ottenuti dal modello.

Tuttavia, per una valutazione più oggettiva delle prestazioni, è opportuno calcolare l'mAP sul dataset di test.

Per farlo è sufficiente richiamare il metodo [**val**](https://docs.ultralytics.com/reference/engine/model/#ultralytics.engine.model.Model.val), impostando il parametro *data* con il percorso del file di configurazione del dataset e il paramentro *split* con la stringa "test", per indicare che la valutazione deve essere effettuata sul test set.

In [ ]:
metrics = yolo_detector.val(data="416x416_aug/data.yaml",split="test")

print("mAP@0.5=", float(metrics.box.map50))

L'elevato valore dell'mAP ottenuto sul test set conferma le ottime prestazioni del modello YOLO11 nel riconoscere correttamente i pezzi degli scacchi, una volta completata la fase di *fine-tuning*.

# **Esercizio 2**
Risolvere un altro problema di *object detection* scelto su [**Roboflow**](https://public.roboflow.com/object-detection) modificando il codice visto durante l'esercitazione.